In [ ]:
# !pip install dvc dvc-gdrive
# !pip install torchinfo

In [ ]:
# !git clone https://github.com/vukhai248/CIDIS-Super-Resolution.git
# %cd CIDIS-Super-Resolution
# !dvc pull 


In [ ]:
import os
import math
from tqdm.auto import tqdm
import matplotlib.pyplot as plt 

import numpy as np 
from PIL import Image
import torch
import torch.nn as nn 
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F 

import albumentations as A
from albumentations.pytorch import ToTensorV2

from torchinfo import summary
from sklearn.model_selection import train_test_split

In [ ]:
seed = 42
data_path = '../data/raw/dataset_CIDIS_sisr_x8/thermal' # For local
# data_path = '/content/dataset_CIDIS_sisr_x8/dataset_CIDIS_sisr_x8/thermal' # For colab
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 64
lr = 1e-5
in_channels = 3
out_channels = 3
num_features = 64
num_blocks = 16
res_scale = 0.1
epochs = 50
max_count = 5
log_interval = 5
best_model_path = '../model/EDSR.pth' # For local
best_model_path = '/content/EDSR.pth' # For Google Colab

In [ ]:
def set_seed(seed=42):
    # np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(seed)

In [ ]:
train_path_LR = os.path.join(data_path, 'train', 'LR_x8')
train_path_GT = os.path.join(data_path, 'train', 'GT')
print(train_path_LR)
print(train_path_GT)

In [ ]:
val_path_LR = os.path.join(data_path, 'val', 'LR_x8')
val_path_GT = os.path.join(data_path, 'val', 'GT')
print(val_path_LR)
print(val_path_GT)

In [ ]:
test_path = os.path.join(data_path, 'test', 'sisr_x8', 'LR_x8')
print(test_path)

In [ ]:
class CIDISDataset(Dataset):
    def __init__(self, data_path, transforms=None, split='train'):
        self.split = split
        self.transforms = transforms
        if split != 'test':
            self.LR_dir = os.path.join(data_path, split, 'LR_x8')
            self.GT_dir = os.path.join(data_path, split, 'GT')

            self.LR_images = sorted(os.listdir(self.LR_dir))
            self.GT_images = sorted(os.listdir(self.GT_dir))
        else: 
            self.LR_dir = os.path.join(data_path, split, 'sisr_x8', 'LR_x8')
            self.LR_images = os.listdir(self.LR_dir)

    def __len__(self):
        return len(self.LR_images)

    def __getitem__(self, idx):

        LR_image_path = os.path.join(self.LR_dir, self.LR_images[idx])
        LR_image = Image.open(LR_image_path).convert('RGB')
        LR_image = np.array(LR_image)
        if self.split == 'test':
            return LR_image
        
        GT_image_path = os.path.join(self.GT_dir, self.GT_images[idx])
        GT_image = Image.open(GT_image_path).convert('RGB')
        GT_image = np.array(GT_image)

        if self.transforms:
            transform_image = self.transforms(image=LR_image, GT_image=GT_image)
            LR_image = transform_image['image']
            GT_image = transform_image['GT_image']

        return LR_image, GT_image    


In [ ]:
train_transforms = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2()
], additional_targets={'GT_image': 'image'}, is_check_shapes=False)

val_test_transforms = A.Compose([
    A.Normalize(mean=(0.0, 0.0, 0.0), std=(1.0, 1.0, 1.0), max_pixel_value=255.0),
    ToTensorV2()
], additional_targets={'GT_image': 'image'}, is_check_shapes=False)

In [ ]:
train_dataset = CIDISDataset(data_path, transforms=train_transforms, split='train')
val_dataset = CIDISDataset(data_path, transforms=val_test_transforms, split='val')
test_dataset = CIDISDataset(data_path, transforms=val_test_transforms, split='test')

In [ ]:
LR_IMG, GT_IMG = train_dataset[0]
print(LR_IMG.shape, GT_IMG.shape)

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
lr_img, gt_img = next(iter(train_loader))
print(lr_img.shape, gt_img.shape)

# Hiển thị 1 cặp ảnh đầu tiên trong batch
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(lr_img[0].permute(1, 2, 0))  # Lấy ảnh đầu tiên, CHW -> HWC
axes[0].set_title('LR')
axes[1].imshow(gt_img[0].permute(1, 2, 0))
axes[1].set_title('GT')
plt.show()


## pipeline model 1: EDSR



In [ ]:
class ChannelAttention(nn.Module):
    def __init__(self, num_features, reduction=16):
        super().__init__()
        self.avg = nn.AdaptiveAvgPool2d(1)
        self.conv = nn.Sequential(
            nn.Conv2d(num_features, num_features // reduction, kernel_size=1, padding=0, bias=True),
            nn.ReLU(inplace=True),
            nn.Conv2d(num_features // reduction, num_features, kernel_size=1, padding=0, bias=True),
            nn.Sigmoid()
        )

    def forward(self, x):
        y = self.avg(x) 
        y = self.conv(y)
        return x * y
        

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, num_features, res_scale=0.1):
        super().__init__()
        self.conv1 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)
        self.ChannelAttention = ChannelAttention(num_features, reduction=16)
        self.relu = nn.ReLU()
        self.res_scale = res_scale
    
    def forward(self, x):
        residual = x
        out = self.relu(self.conv1(x))
        out = self.conv2(out)
        out = self.ChannelAttention(out) * self.res_scale + residual
        return out

In [ ]:
class Upsampler(nn.Module):
    """Upsample for x8 SR"""
    def __init__(self, in_channels, scale_factor=8):
        super().__init__()
        self.upsampler = nn.ModuleList([
            nn.Sequential(
                nn.Conv2d(in_channels, in_channels * 4, kernel_size=3, padding=1),
                nn.PixelShuffle(2),
                nn.ReLU(inplace=True)
            ) for _ in range(int(math.log2(scale_factor)))
        ])
    
    def forward(self, x):
        for block in self.upsampler:
            x = block(x)
        return x

In [ ]:
class EDSR(nn.Module):
    def __init__(self, in_channels, out_channels, num_features, num_blocks, res_scale=0.1):
        super().__init__()

        out_channels = out_channels if out_channels is not None else in_channels

        # Head
        self.head = nn.Conv2d(in_channels, num_features, kernel_size=3, padding=1)

        # Body (Stack ResBlock)
        self.StackResBlock = nn.ModuleList([
            ResBlock(num_features, res_scale=res_scale) for _ in range(num_blocks)
        ])

        # Upsampler 
        self.upsampler = Upsampler(num_features, scale_factor=8)

        # Additional upsampling conv for fine adjustment
        self.upsample_conv = nn.Conv2d(num_features, num_features, kernel_size=3, padding=1)

        # Tail
        self.tail = nn.Conv2d(num_features, out_channels, kernel_size=3, padding=1)

    def forward(self, x):
        x = self.head(x)
        residual = x
        for block in self.StackResBlock:
            x = block(x)
        x = x + residual
        x = self.upsampler(x)
        x = self.upsample_conv(x)
        x = self.tail(x)
        return x

In [ ]:
model = EDSR(in_channels, out_channels, num_features, num_blocks, res_scale).to(device)
summary(model, next(iter(train_loader))[0].shape[1:])

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.L1Loss()

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.1,
    patience=5,
    min_lr=1e-6
)

In [ ]:
def train(model, optimizer, scheduler, criterion, train_loader, val_loader, epochs, max_count, best_model_path, log_interval, device='cpu'):

    train_loss = []
    val_loss = []
    best_val_loss = 100000
    max_count = max_count
    break_count = 0

    for epoch in tqdm(range(epochs), desc='Current epoch/Total epoch', position=0):

        train_epoch_loss = 0
        val_epoch_loss = 0
        
        # Training parse
        model.train()
        for LR_image, GT_image in tqdm(train_loader, desc='Training batch', position=1, leave=False):
            LR_image = LR_image.to(device)
            GT_image = GT_image.to(device)

            optimizer.zero_grad()

            predictions = model(LR_image)

            loss = criterion(predictions, GT_image)
            train_epoch_loss += loss.item()

            loss.backward()
            optimizer.step()

        train_loss.append(train_epoch_loss / len(train_loader))

        # Eval parse
        model.eval()
        with torch.no_grad():
            for LR_image, GT_image in tqdm(val_loader, desc='Valid batch', position=1, leave=False):
                LR_image = LR_image.to(device)
                GT_image = GT_image.to(device)

                predictions = model(LR_image)

                loss = criterion(predictions, GT_image)
                val_epoch_loss += loss.item()

        val_loss.append(val_epoch_loss / len(val_loader))

        scheduler.step(val_loss[-1])

        if val_loss[-1] < best_val_loss:
            break_count = 0
            best_val_loss = val_loss[-1]
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val_loss': val_loss[-1],
            }, best_model_path)
        else:
            break_count += 1
        
        if break_count >= max_count:
            print(f'Break at Epoch: {epoch} with best validation loss: {val_loss[-1]}')
            break

        if epoch % log_interval == 0:
            print(f'| Epoch {epoch} |  Train Loss {train_loss[-1]} | Val Loss {val_loss[-1]} |')

    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print('Training complete and load best model')
    metrics = {
        'train_loss': train_loss,
        'val_loss': val_loss
    }
    return model, metrics

In [ ]:
model, metrics = train(model, 
      optimizer, 
      scheduler, 
      criterion, 
      train_loader, 
      val_loader, 
      epochs, 
      max_count, 
      best_model_path, 
      log_interval,
      device)

In [ ]:
def visualizer(metrics):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Biểu đồ riêng
    axes[0].plot(metrics['train_loss'], color='blue', label='Train')
    axes[0].set_title('Training Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    axes[1].plot(metrics['val_loss'], color='orange', label='Validation')
    axes[1].set_title('Validation Loss')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()

    # Biểu đồ chung 
    fig2, ax = plt.subplots(figsize=(8, 5))
    ax.plot(metrics['train_loss'], color='blue', label='Train')
    ax.plot(metrics['val_loss'], color='orange', label='Validation')
    ax.set_title('Train vs Validation Loss')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss')
    ax.legend()

    plt.tight_layout()
    plt.show()

visualizer(metrics)

In [ ]:
def test(model, test_loader, best_model_path, device):
    checkpoint = torch.load(best_model_path, map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    with torch.no_grad():
        for LR_images in tqdm(test_loader, desc='Test batch', leave=False):
            LR_images = LR_images.to(device)
            predictions = model(LR_images)  # [B, 3, H*8, W*8]

            # Visual từng ảnh trong batch
            for i in range(len(LR_images)):
                lr_img = LR_images[i].cpu().permute(1, 2, 0).numpy()   # [H, W, 3]
                sr_img = predictions[i].cpu().permute(1, 2, 0).numpy() # [H*8, W*8, 3]
                
                # Clip về [0, 1] tránh artifact khi hiển thị
                lr_img = lr_img.clip(0, 1)
                sr_img = sr_img.clip(0, 1)

                fig, axes = plt.subplots(1, 2, figsize=(14, 5))
                axes[0].imshow(lr_img)
                axes[0].set_title(f'LR [{LR_images[i].shape[1]}x{LR_images[i].shape[2]}]')
                axes[0].axis('off')

                axes[1].imshow(sr_img)
                axes[1].set_title(f'SR x8 [{predictions[i].shape[1]}x{predictions[i].shape[2]}]')
                axes[1].axis('off')

                plt.tight_layout()
                plt.show()

In [ ]:
test(model, test_loader, best_model_path, device)